# Genetic Algorithm: Animal Camouflage RGB

**Goal:** Find an RGB color close to the environment color **(34, 186, 56)**.

Each candidate animal color is a chromosome with three genes: **Red, Green, Blue**. Each gene is an integer from **0 to 255**. We use Euclidean distance (ED) as the measure of color difference, a population of 100, and then compare the genetic algorithm with brute force.

This is a simple color-matching model: it finds a color that matches the target RGB, rather than modeling real animal biology.

## 1. Setup the target color and GA settings

In [16]:
import random
import math
import time
from IPython.display import display, HTML

TARGET_COLOR = [34, 186, 56]
POPULATION_SIZE = 100
MAX_GENERATIONS = 250
CROSSOVER_RATE = 0.80
MUTATION_RATE = 0.12
MUTATION_STEP = 25

random.seed(42)

print("Target RGB color:", TARGET_COLOR)
print("Population size:", POPULATION_SIZE)
print("Maximum generations:", MAX_GENERATIONS)

Target RGB color: [34, 186, 56]
Population size: 100
Maximum generations: 250


In [5]:
def show_color(color, label):
    red, green, blue = color
    box = (
        f'<div style="display:inline-block; padding:12px 18px; '
        f'background:rgb({red},{green},{blue}); color:white; '
        f'border:1px solid #333; border-radius:5px;">'
        f'{label}: RGB({red}, {green}, {blue})</div>'
    )
    display(HTML(box))

show_color(TARGET_COLOR, "Environment target")

## 2. Euclidean distance (ED)

ED measures how far a candidate RGB color is from the target. A smaller distance means a closer color match.

In [6]:
def euclidean_distance(color1, color2):
    total = 0

    for i in range(3):
        difference = color1[i] - color2[i]
        total += difference * difference

    return math.sqrt(total)

print("Example ED:", round(euclidean_distance([30, 180, 60], TARGET_COLOR), 2))

Example ED: 8.25


## 3. Fitness function

The best possible distance is 0. Fitness is scaled from 0 to 1, where **1 means an exact RGB match**.

In [7]:
MAX_DISTANCE = math.sqrt(
    sum(max(channel, 255 - channel) ** 2 for channel in TARGET_COLOR)
)

def fitness(color):
    distance = euclidean_distance(color, TARGET_COLOR)
    return 1 - (distance / MAX_DISTANCE)

print("Fitness of target color:", fitness(TARGET_COLOR))
print("Fitness of [0, 0, 0]:", round(fitness([0, 0, 0]), 4))

Fitness of target color: 1.0
Fitness of [0, 0, 0]: 0.4378


## 4. Create the initial population

Each chromosome contains three integer genes: `[Red, Green, Blue]`. The population contains 100 random colors.

In [8]:
def create_chromosome():
    return [random.randint(0, 255) for _ in range(3)]

population = [create_chromosome() for _ in range(POPULATION_SIZE)]

print("Population members:", len(population))
print("First five RGB chromosomes:")
for chromosome in population[:5]:
    print(chromosome)

Population members: 100
First five RGB chromosomes:
[57, 12, 140]
[125, 114, 71]
[52, 44, 216]
[16, 15, 47]
[111, 119, 13]


## 5. Roulette-wheel selection

Colors with higher fitness have a higher chance of becoming parents.

In [9]:
def roulette_wheel_selection(population, fitness_values):
    weights = [value + 0.001 for value in fitness_values]
    total_weight = sum(weights)
    pick = random.random() * total_weight
    running_total = 0

    for chromosome, weight in zip(population, weights):
        running_total += weight
        if running_total >= pick:
            return chromosome[:]

    return population[-1][:]

## 6. Crossover

Single-point crossover swaps part of two parent chromosomes to create children.

In [10]:
def crossover(parent1, parent2):
    if random.random() >= CROSSOVER_RATE:
        return parent1[:], parent2[:]

    point = random.randint(1, 2)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2

print("Example crossover:")
print(crossover([10, 100, 20], [200, 30, 220]))

Example crossover:
([10, 30, 220], [200, 100, 20])


## 7. Mutation

A gene sometimes changes by a small random amount. Values are clipped to stay in the valid RGB range of 0–255.

In [11]:
def mutate(chromosome):
    child = chromosome[:]

    for i in range(3):
        if random.random() < MUTATION_RATE:
            change = random.randint(-MUTATION_STEP, MUTATION_STEP)
            child[i] = max(0, min(255, child[i] + change))

    return child

print("Example mutation:", mutate([34, 186, 56]))

Example mutation: [34, 186, 56]


## 8. Run the genetic algorithm

The best chromosome is copied into each new generation (**elitism**) so it is not lost. The run stops if an exact match is found.

In [12]:
ga_start = time.perf_counter()

best_overall = None
best_overall_distance = float("inf")
fitness_history = []
generations_used = 0

for generation in range(MAX_GENERATIONS):
    fitness_values = [fitness(chromosome) for chromosome in population]

    best_index = fitness_values.index(max(fitness_values))
    current_best = population[best_index][:]
    current_distance = euclidean_distance(current_best, TARGET_COLOR)

    if current_distance < best_overall_distance:
        best_overall = current_best[:]
        best_overall_distance = current_distance

    generations_used = generation + 1
    fitness_history.append((generations_used, best_overall[:], best_overall_distance))

    if best_overall_distance == 0:
        break

    next_population = [best_overall[:]]  # Keep the best candidate

    while len(next_population) < POPULATION_SIZE:
        parent1 = roulette_wheel_selection(population, fitness_values)
        parent2 = roulette_wheel_selection(population, fitness_values)
        child1, child2 = crossover(parent1, parent2)

        next_population.append(mutate(child1))
        if len(next_population) < POPULATION_SIZE:
            next_population.append(mutate(child2))

    population = next_population

ga_time = time.perf_counter() - ga_start

print("Genetic algorithm finished.")
print("Generations used:", generations_used)
print("Best RGB found:", best_overall)
print("ED from target:", round(best_overall_distance, 4))
print("Fitness:", round(fitness(best_overall), 6))
print("Time (seconds):", round(ga_time, 4))
show_color(best_overall, "GA result")

Genetic algorithm finished.
Generations used: 117
Best RGB found: [34, 186, 56]
ED from target: 0.0
Fitness: 1.0
Time (seconds): 0.1282


## 9. Fitness progress

Show the best distance at selected generations. A distance of 0 means the target was found exactly.

In [13]:
for generation_number, color, distance in fitness_history:
    if generation_number == 1 or generation_number % 10 == 0 or distance == 0:
        print("Generation:", generation_number,
              "Best RGB:", color,
              "ED:", round(distance, 4))

Generation: 1 Best RGB: [31, 160, 29] ED: 37.6032
Generation: 10 Best RGB: [31, 183, 55] ED: 4.3589
Generation: 20 Best RGB: [33, 184, 59] ED: 3.7417
Generation: 30 Best RGB: [34, 184, 55] ED: 2.2361
Generation: 40 Best RGB: [34, 184, 56] ED: 2.0
Generation: 50 Best RGB: [34, 187, 56] ED: 1.0
Generation: 60 Best RGB: [34, 187, 56] ED: 1.0
Generation: 70 Best RGB: [34, 187, 56] ED: 1.0
Generation: 80 Best RGB: [34, 187, 56] ED: 1.0
Generation: 90 Best RGB: [34, 187, 56] ED: 1.0
Generation: 100 Best RGB: [34, 187, 56] ED: 1.0
Generation: 110 Best RGB: [34, 187, 56] ED: 1.0
Generation: 117 Best RGB: [34, 186, 56] ED: 0.0


## 10. Brute-force search

There are `256 ** 3 = 16,777,216` possible RGB colors. Brute force checks colors in order and stops as soon as it finds ED = 0, because zero is the lowest possible distance. If no exact match existed, the loops would examine every color.

In [14]:
brute_start = time.perf_counter()

brute_best_color = None
brute_best_distance_squared = float("inf")
checked_colors = 0

for red in range(256):
    for green in range(256):
        for blue in range(256):
            checked_colors += 1
            dr = red - TARGET_COLOR[0]
            dg = green - TARGET_COLOR[1]
            db = blue - TARGET_COLOR[2]
            distance_squared = dr * dr + dg * dg + db * db

            if distance_squared < brute_best_distance_squared:
                brute_best_distance_squared = distance_squared
                brute_best_color = [red, green, blue]

            if distance_squared == 0:
                break
        if brute_best_distance_squared == 0:
            break
    if brute_best_distance_squared == 0:
        break

brute_time = time.perf_counter() - brute_start
brute_best_distance = math.sqrt(brute_best_distance_squared)

print("Brute-force RGB:", brute_best_color)
print("ED from target:", brute_best_distance)
print("Colors checked before finding exact match:", checked_colors)
print("Total possible RGB colors:", 256 ** 3)
print("Time (seconds):", round(brute_time, 4))
show_color(brute_best_color, "Brute-force result")

Brute-force RGB: [34, 186, 56]
ED from target: 0.0
Colors checked before finding exact match: 2275897
Total possible RGB colors: 16777216
Time (seconds): 1.1694


## 11. Compare both approaches

In [15]:
print("TARGET RGB:", TARGET_COLOR)
print()
print("Genetic algorithm")
print("  RGB:", best_overall)
print("  ED :", round(best_overall_distance, 4))
print("  Time:", round(ga_time, 4), "seconds")
print()
print("Brute force")
print("  RGB:", brute_best_color)
print("  ED :", round(brute_best_distance, 4))
print("  Time:", round(brute_time, 4), "seconds")
print()
print("Same result?", best_overall == brute_best_color)
print("Brute force checked", checked_colors, "of", 256 ** 3, "possible colors before it found ED = 0.")

TARGET RGB: [34, 186, 56]

Genetic algorithm
  RGB: [34, 186, 56]
  ED : 0.0
  Time: 0.1282 seconds

Brute force
  RGB: [34, 186, 56]
  ED : 0.0
  Time: 1.1694 seconds

Same result? True
Brute force checked 2275897 of 16777216 possible colors before it found ED = 0.
